# Numerical Diagnostics

In [1]:
import polars as pl
import numpy as np
from datascience import make_array
import hvplot.polars
import holoviews as hv
from IPython.display import display
path_data = '../../../assets/data/'
from scipy import stats

In addition to visualization, we can use numerical properties of residuals to assess the quality of regression. We will not prove these properties mathematically. Rather, we will observe them by computation and see what they tell us about the regression.

All of the facts listed below hold for all shapes of scatter plots, whether or not they are linear.

In [2]:
family_heights = pl.read_csv(path_data + 'family_heights.csv', infer_schema_length=10000)
heights = family_heights.select('midparentHeight', 'childHeight')
heights = heights.rename({'midparentHeight': 'MidParent', 'childHeight': 'Child'})
dugong = pl.read_csv(path_data + 'dugongs.csv')
dugong = dugong.select('Length', pl.exclude('Length'))
hybrid = pl.read_csv(path_data + 'hybrid.csv')

In [3]:
def standard_units(x):
    return (x - x.mean())/x.std(ddof=0)

def correlation(table, x, y):
    x_in_standard_units = standard_units(table.get_column(x))
    y_in_standard_units = standard_units(table.get_column(y))
    return (x_in_standard_units * y_in_standard_units).mean()

def slope(table, x, y):
    r = correlation(table, x, y)
    return r * table.get_column(y).std(ddof=0)/table.get_column(x).std(ddof=0)

def intercept(table, x, y):
    a = slope(table, x, y)
    return table.get_column(y).mean() -  a * table.get_column(x).mean()

def fit(table, x, y):
    a = slope(table, x, y)
    b = intercept(table, x, y)
    return a * table.get_column(x) + b

def residual(table, x, y):
    return table.get_column(y) - fit(table, x, y)

def scatter_fit(table, x, y):
    # POLARS_CHANGE: hvplot scatter with a HoloViews Curve overlay for the regression line.
    return (
        table.hvplot.scatter(x=x, y=y, s=15, color="#002fa7")
        * hv.Curve((table.get_column(x).to_numpy(), fit(table, x, y).to_numpy())).opts(color="black", line_width=5)  # dark outline for contrast
        * hv.Curve((table.get_column(x).to_numpy(), fit(table, x, y).to_numpy())).opts(color="#FFFF00", line_width=4)
    ).opts(xlabel=x, ylabel=y)
    
def residual_plot(table, x, y):
    x_array = table.get_column(x)
    t = pl.DataFrame([
            x_array,
            residual(table, x, y).alias('residuals')
        ])
    xlims = make_array(x_array.min(), x_array.max())
    return (
        t.hvplot.scatter(x=x, y='residuals', color='red')
        * hv.Curve((xlims, make_array(0, 0))).opts(color='darkblue', line_width=4)
    ).opts(title='Residual Plot')

def regression_diagnostic_plots(table, x, y):
    # POLARS_CHANGE: HoloViews charts are returned, so display shows both figures in order.
    display(scatter_fit(table, x, y))
    display(residual_plot(table, x, y))   

In [4]:
heights = heights.with_columns(
        fit(heights, 'MidParent', 'Child').alias('Fitted Value'),
        residual(heights, 'MidParent', 'Child').alias('Residual')
    )

## Residual Plots Show No Trend
**For every linear regression, whether good or bad, the residual plot shows no trend. Overall, it is flat. In other words, the residuals and the predictor variable are uncorrelated.** 

You can see this in all the residual plots above. We can also calculate the correlation between the predictor variable and the residuals in each case.



In [5]:
correlation(heights, 'MidParent', 'Residual')

1.1769077266395798e-15

That doesn't look like zero, but it is a tiny number that is 0 apart from rounding error due to computation. Here it is again, correct to 10 decimal places. A minus sign, when it appears, is because of the rounding error above. (POLARS_CHANGE: Polars adds the values in a different order than NumPy, so the tiny error and the sign of the rounded `0.0` can differ from the Data 8 output.)

In [6]:
round(correlation(heights, 'MidParent', 'Residual'), 10)

0.0

In [7]:
dugong = dugong.with_columns(
       fit(dugong, 'Length', 'Age').alias('Fitted Value'),
       residual(dugong, 'Length', 'Age').alias('Residual')
)
round(correlation(dugong, 'Length', 'Residual'), 10)

0.0

## Average of Residuals
**No matter what the shape of the scatter diagram, the average of the residuals is 0.**

This is analogous to the fact that if you take any list of numbers and calculate the list of deviations from average, the average of the deviations is 0.

In all the residual plots above, you have seen the horizontal line at 0 going through the center of the plot. That is a visualization of this fact.

As a numerical example, here is the average of the residuals in the regression of children's heights based on midparent heights.

In [8]:
round(heights.get_column('Residual').mean(), 10)

-0.0

The same is true of the average of the residuals in the regression of the age of dugongs on their length. The mean of the residuals is 0, apart from rounding error. 

In [9]:
round(dugong.get_column('Residual').mean(), 10)

-0.0

## SD of the Residuals
**No matter what the shape of the scatter plot, the SD of the residuals is a fraction of the SD of the response variable. The fraction is  $\sqrt{1-r^2}$.**

$$
\text{SD of residuals} ~=~ \sqrt{1 - r^2} \cdot \text{SD of }y
$$

We will soon see how this measures the accuracy of the regression estimate. But first, let's confirm it by example. 

In the case of children's heights and midparent heights, the SD of the residuals is about 3.39 inches.

In [10]:
heights.get_column('Residual').std(ddof=0)

3.388079916395342

That's the same as $\sqrt{1-r^2}$ times the SD of response variable: 

In [11]:
r = correlation(heights, 'MidParent', 'Child')
np.sqrt(1 - r**2) * heights.get_column('Child').std(ddof=0)

3.3880799163953434

The same is true for the regression of mileage on acceleration of hybrid cars. The correlation $r$ is negative (about -0.5), but $r^2$ is positive and therefore $\sqrt{1-r^2}$ is a fraction.

In [12]:
r = correlation(hybrid, 'acceleration', 'mpg')
r

-0.5060703843771186

In [13]:
hybrid = hybrid.with_columns(
     fit(hybrid, 'acceleration', 'mpg').alias('fitted mpg'),
     residual(hybrid, 'acceleration', 'mpg').alias('residual')
)
hybrid.get_column('residual').std(ddof=0), np.sqrt(1 - r**2)*hybrid.get_column('mpg').std(ddof=0)

(9.43273683343029, 9.4327368334302903)

Now let us see how the SD of the residuals is a measure of how good the regression is. Remember that the average of the residuals is 0. Therefore the smaller the SD of the residuals is, the closer the residuals are to 0. In other words, if the SD of the residuals is small, the overall size of the errors in regression is small.

The extreme cases are when $r=1$ or $r=-1$. In both cases, $\sqrt{1-r^2} = 0$. Therefore the residuals have an average of 0 and an SD of 0 as well, and therefore the residuals are all equal to 0. The regression line does a perfect job of estimation. As we saw earlier in this chapter, if $r = \pm 1$, the scatter plot is a perfect straight line and is the same as the regression line, so indeed there is no error in the regression estimate. 

But usually $r$ is not at the extremes. If $r$ is neither $\pm 1$ nor 0, then $\sqrt{1-r^2}$ is a proper fraction, and the rough overall size of the error of the regression estimate is somewhere between 0 and the SD of $y$.

The worst case is when $r = 0$. Then $\sqrt{1-r^2} =1$, and the SD of the residuals is equal to the SD of $y$. This is consistent with the observation that if $r=0$ then the regression line is a flat line at the average of $y$. In this situation, the root mean square error of regression is the root mean squared deviation from the average of $y$, which is the SD of $y$. In practical terms, if $r = 0$ then there is no linear association between the two variables, so there is no benefit in using linear regression.

## Another Way to Interpret $r$
We can rewrite the result above to say that no matter what the shape of the scatter plot,

$$
\frac{\text{SD of residuals}}{\text{SD of }y} ~=~ \sqrt{1-r^2}
$$

A complementary result is that no matter what the shape of the scatter plot, the SD of the fitted values is a fraction of the SD of the observed values of $y$. The fraction is $\vert r \vert$.

$$
\frac{\text{SD of fitted values}}{\text{SD of }y} ~=~ \vert r \vert
$$

To see where the fraction comes in, notice that the fitted values are all on the regression line whereas the observed values of $y$ are the heights of all the points in the scatter plot and are more variable.

In [14]:
scatter_fit(heights, 'MidParent', 'Child')

Scatterplot with 'MidParent' on the x-axis and 'Child' on the y-axis. The data points are in dark blue and have a weak to moderate positive association. A yellow line is drawn through the center of the data with a positive slope.

The fitted values range from about 64 to about 71, whereas the heights of all the children are quite a bit more variable, ranging from about 55 to 80.

To verify the result numerically, we just have to calculate both sides of the identity.

In [15]:
correlation(heights, 'MidParent', 'Child')

0.32094989606395874

Here is ratio of the SD of the fitted values and the SD of the observed values of birth weight:

In [16]:
heights.get_column('Fitted Value').std(ddof=0)/heights.get_column('Child').std(ddof=0)

0.32094989606395913

The ratio is equal to $r$, confirming our result.

Where does the absolute value come in? First note that as SDs can't be negative, nor can a ratio of SDs. So what happens when $r$ is negative? The example of fuel efficiency and acceleration will show us.

In [17]:
correlation(hybrid, 'acceleration', 'mpg')

-0.5060703843771186

In [18]:
hybrid.get_column('fitted mpg').std(ddof=0)/hybrid.get_column('mpg').std(ddof=0)

0.5060703843771187

The ratio of the two SDs is $\vert r \vert$. 

A more standard way to express this result is to recall that 

$$
\text{variance} ~=~ \text{mean squared deviation from average} ~=~ \text{SD}^2
$$

and therefore, by squaring both sides of our result,

$$
\frac{\text{variance of fitted values}}{\text{variance of }y} ~=~ r^2
$$